# 🚀 LISAN AI: Online GPU Fine-Tuning & Model Training
### End-to-End Multilingual Speech-to-Speech Translation for Ethiopian Languages

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ermizamr/LISAN/blob/main/notebooks/lisan_online_gpu_training.ipynb)

**Supported Languages:**
- **Amharic (አማርኛ)** (`amh_Ethi`)
- **Afaan Oromo** (`gaz_Latn`)
- **Tigrinya (ትግርኛ)** (`tir_Ethi`)
- **Somali (Soomaali)** (`som_Latn`)
- **English** (`eng_Latn`)

**What this notebook trains & exports:**
1. 🌐 **Machine Translation (NMT)**: LoRA Fine-Tuning on `facebook/nllb-200-distilled-600M` with 68,000+ curated Ethiopian sentence pairs $\rightarrow$ automatic weight merging $\rightarrow$ CTranslate2 INT8 quantization.
2. 🎙️ **Speech-to-Text (STT)**: Speech dataset preparation (Google FLEURS / Common Voice) + Whisper LoRA fine-tuning + WER & latency benchmarking.
3. 🔊 **Text-to-Speech (TTS)**: Meta MMS-TTS neural voice verification & RTF (Real-Time Factor) benchmarking across all 5 languages.
4. 📦 **One-Click Download**: Automatically zips and downloads all trained weights to your laptop for instant offline use in the Flutter app.

--- 
## 1. Verify GPU Acceleration
Ensure your runtime has a GPU allocated (in Google Colab: **Runtime** $\rightarrow$ **Change runtime type** $\rightarrow$ **T4 GPU** or **A100 GPU**).

In [ ]:
!nvidia-smi

import torch
assert torch.cuda.is_available(), "ERROR: No GPU detected! Please select a GPU in Runtime -> Change runtime type."
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f"\n✓ GPU Detected: {gpu_name} ({vram_gb:.2f} GB VRAM)")

--- 
## 2. Clone Repository & Install Training Dependencies

In [ ]:
import os
if not os.path.exists("LISAN"):
    !git clone https://github.com/ermizamr/LISAN.git
    %cd LISAN
else:
    %cd LISAN
    !git pull origin main

# Install GPU-accelerated training packages
!pip install -q -r training_requirements.txt

--- 
## 3. Run Environment & Hardware Diagnostics

In [ ]:
!python scripts/check_gpu_env.py

--- 
## 4. Pillar 1: Machine Translation (NMT) LoRA Training & CTranslate2 INT8 Export
Trains on 68,000+ parallel sentence pairs, merges the LoRA adapter into the base NLLB model, and quantizes to CTranslate2 INT8 format.

In [ ]:
# Execute NMT training, weight merge, and CTranslate2 export
!python scripts/train_nllb_lora.py \
  --batch_size 16 \
  --grad_accum 2 \
  --num_epochs 3 \
  --learning_rate 2e-4 \
  --merge_and_export \
  --eval_benchmarks

--- 
## 5. Pillar 2: Speech-to-Text (STT) Data Preparation & Whisper Fine-Tuning

In [ ]:
# 1. Download and format Google FLEURS speech datasets for Amharic, Oromo, and Somali
!python scripts/prepare_stt_data.py --source fleurs --max_samples_per_lang 300

# 2. Fine-tune Whisper with LoRA on Ethiopian audio
!python scripts/train_whisper_lora.py \
  --base_model openai/whisper-small \
  --batch_size 8 \
  --grad_accum 4 \
  --num_epochs 3 \
  --merge_and_save

--- 
## 6. STT Accuracy & Latency Benchmarks (WER Evaluation)

In [ ]:
!python scripts/evaluate_stt.py

--- 
## 7. Pillar 3: Text-to-Speech (TTS) Verification & RTF Benchmarking
Tests neural speech synthesis across Amharic, Afaan Oromo, Tigrinya, Somali, and English.

In [ ]:
!python scripts/benchmark_tts.py

--- 
## 8. Package & Download Trained Models to Your Local Machine
Compresses all fine-tuned CTranslate2 INT8 weights, Whisper weights, and synthesized audio into a single `.zip` file and initiates browser download.

In [ ]:
# Create zip archive of trained models
!zip -r lisan_trained_models.zip models_optimized/ output_tts/

# Download automatically if running in Google Colab
try:
    from google.colab import files
    files.download('lisan_trained_models.zip')
    print('✓ Download started automatically!')
except ImportError:
    print('✓ Archive created at: lisan_trained_models.zip')
    print('Download it via Jupyter file browser or terminal scp.')